# Choosing qubits: benchmarking a processor you cannot see inside

| | |
|---|---|
| **Level** | Advanced |
| **Time** | About 90 minutes |
| **Prerequisites** | Clifford gates; benchmarking ideas such as randomized benchmarking are helpful |
| **Default devices** | IQM Garnet, with circuits run exactly as written (see below) |
| **Hardware jobs** | 8 for the depth sweep, 6 for qubit ranking, 2 for the payoff |
| **Approximate cost** | about 2,200 credits on Garnet at the default shots (820 for the sweep, 1,400 for ranking and payoff) |
| **Hardware notes** | Tested 25 September 2026 on IQM Garnet, verbatim, at 100 shots. Mirror survival fell from 0.82 at 4 two-qubit gates to 0.74 at 16. The ranking separated the qubits (survival 0.98 for the best, 0.79 for the worst). BB84 gave a QBER of 1.0% on the best four and 1.75% on the worst four. Without the verbatim mode, the compilers of both default devices removed the mirror circuits entirely. |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*QUEST intermediate and advanced series: Systems, Hardware and Engineering*

The other notebooks in this series pick a device by name and submit. That hides a decision. Qubits on a real device are not all equally good: some are clearly worse than others, the ranking changes between calibrations, and a circuit placed on the wrong four qubits can return noise while the same circuit on the right four returns a usable result. The algorithm stays the same; only the placement changes.

The obvious way to choose is to read the vendor's calibration data. That works within one vendor's system but is hard to compare across vendors: the numbers are measured by different protocols, defined differently, and refreshed on different schedules.

So we measure the devices ourselves, with circuits that need no calibration file and no reference device. Mirror circuits give a number that means the same thing on a superconducting chip and on a trapped-ion device, because it is defined by the circuits we ran. We check the method on a simulator with a known error rate, run it on real devices, rank the qubits of one device, and then run the BB84 circuit from the Cryptography notebook on the best and worst qubits to see what the ranking is worth.

**Learning objectives**

1. State what a device interface can and cannot tell you about qubit quality.
2. Build randomized mirror circuits and predict their ideal output without simulating them.
3. Validate a benchmark by injecting a known error rate and recovering it.
4. Explain why a benchmark must be protected from the compiler.
5. Rank the qubits of a real device by measured survival and choose a subset.
6. Measure what that choice is worth by running a real protocol on the best and worst subsets.

**Background needed:** single- and two-qubit gates, the Clifford group (generated by H, S and CNOT), and how a depolarizing channel acts. The BB84 circuit is restated here, so the Cryptography notebook is useful but not required.


## What the device interface reports

We start with what the SDK reports about each device, because the gap between that and what we need motivates the rest of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Clifford, Pauli
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

rng = np.random.default_rng(11)
sim = AerSimulator(seed_simulator=42)

print("Setup complete.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
HW_SHOTS = 500
RANK_SHOTS = 1000
PAYOFF_SHOTS = 1000
QUEST_JOB_TAGS = {"quest": "noise-benchmark"}   # labels this notebook's hardware jobs for QUEST usage statistics


In [ ]:
provider = QbraidProvider()

# Devices are named by qBraid QRN. The README lists devices, prices and availability.
BACKENDS = {
    'Rigetti Cepheus': 'rigetti:rigetti:qpu:cepheus-1-108q',
    'IQM Garnet':      'aws:iqm:qpu:garnet',
    # 'AQT IBEX Q1':   'aws:aqt:qpu:ibex-q1',   # trapped ion; runs in scheduled windows; 2.35 credits per shot
}

COLORS = {
    'Rigetti Cepheus': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT IBEX Q1':     '#2d7a4f',
}

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}

meta = pd.DataFrame([
    {'Backend': name, **{k: v for k, v in dev.metadata().items()
                         if k in ('num_qubits', 'provider_name', 'paradigm',
                                  'status', 'queue_depth', 'average_queue_time')}}
    for name, dev in devices.items()
]).set_index('Backend')
meta

import re

_NOT_GATES = {"openqasm", "include", "bit", "qubit", "box", "measure", "declare", "pragma",
              "b", "c", "meas", "ro", "barrier", "fence", "delay", "halt", "reset", "defcal", "cal"}

def _count_gates(program_text):
    """(all gates, two-qubit gates) in a compiled OpenQASM or Quil program."""
    total = two = 0
    for line in program_text.splitlines():
        m = re.match(r"\s*([A-Za-z_]+)", line)
        if not m or m.group(1).lower() in _NOT_GATES:
            continue
        total += 1
        if m.group(1).lower() in ("cz", "cx", "cnot", "iswap", "xy", "cphase", "ecr", "ms", "zz"):
            two += 1
    return total, two

def check_what_ran(job, circuit):
    """Compare the circuit we sent with the program the device actually ran.

    Device compilers rewrite circuits before running them. Usually that only
    changes the gate names, but a compiler can also remove gates that cancel,
    such as a circuit followed by its inverse. This prints both gate counts.
    """
    ops = [inst.operation for inst in circuit.data if inst.operation.name not in ("barrier", "measure")]
    sent_total, sent_two = len(ops), sum(1 for op in ops if op.num_qubits == 2)
    try:
        program = job.client.get_job_compiled_program(job.id)
    except Exception as err:
        print(f"  could not fetch the compiled program ({type(err).__name__}); check skipped")
        return None
    ran_total, ran_two = _count_gates(getattr(program, "data", str(program)))
    print(f"  gates sent {sent_total} ({sent_two} two-qubit); device ran {ran_total} ({ran_two} two-qubit)")
    removed = (sent_two and ran_two < 0.5 * sent_two) or (sent_total >= 10 and ran_total < 0.5 * sent_total)
    if removed:
        print("  WARNING: the compiler removed most of the gates. "
              "This result does not measure the circuit you built.")
    return not removed


# ---- Running circuits exactly as written ----------------------------------
# A device's compiler removes gates that cancel, such as a circuit followed by
# its own inverse, even across barriers. For this notebook that would erase the
# experiment. IQM Garnet (through Amazon Braket) accepts "verbatim" programs:
# native gates on named physical qubits, run gate for gate. run_exactly() uses
# that on Garnet. Rigetti has no such mode through qBraid, so there it submits
# normally, and check_what_ran() reports whether the circuit survived.
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qbraid_core.services.runtime.schemas import Program

GARNET_EDGES = [(3, 4), (3, 8), (4, 5), (8, 9), (8, 13), (9, 10), (9, 14), (10, 11), (10, 15),
                (11, 12), (11, 16), (12, 17), (13, 14), (14, 15), (14, 18), (15, 16), (15, 19),
                (16, 17), (16, 20), (18, 19), (19, 20)]
GARNET_PATH = [14, 15, 16, 17, 12, 11, 10, 9]    # eight connected qubits in a line


def runs_exactly(device_id):
    """True for devices where run_exactly() can bypass the compiler."""
    return device_id.startswith('aws:iqm:')


def garnet_verbatim(qc, physical=GARNET_PATH):
    """qc in IQM's native gates (prx, cz) on the given physical qubits, as a verbatim program."""
    phys = list(physical)[:qc.num_qubits]
    index = {p: i for i, p in enumerate(phys)}
    links = [(index[a], index[b]) for a, b in GARNET_EDGES if a in index and b in index]
    cmap = CouplingMap(links + [(b, a) for a, b in links])
    # optimization_level=0 translates gates without cancelling or merging any.
    native = transpile(qc, basis_gates=['r', 'cz'], coupling_map=cmap,
                       initial_layout=list(range(len(phys))), optimization_level=0,
                       seed_transpiler=1)
    body, reads = [], []
    for inst in native.data:
        qs = [phys[native.find_bit(q).index] for q in inst.qubits]
        name = inst.operation.name
        if name == 'r':
            theta, phi = (float(p) for p in inst.operation.params)
            body.append(f"prx({theta:.12f}, {phi:.12f}) ${qs[0]};")
        elif name == 'cz':
            body.append(f"cz ${qs[0]}, ${qs[1]};")
        elif name == 'measure':
            reads.append(f"b[{native.find_bit(inst.clbits[0]).index}] = measure ${qs[0]};")
        elif name != 'barrier':
            raise ValueError(f"unexpected gate after translation: {name}")
    return (f"OPENQASM 3.0;\nbit[{qc.num_clbits}] b;\n#pragma braket verbatim\nbox{{\n"
            + "\n".join(body) + "\n}\n" + "\n".join(reads) + "\n")


def run_exactly(name, device, qc, shots, physical=GARNET_PATH):
    """Submit qc; return (counts, ran_as_written). On Garnet it runs gate for gate."""
    if runs_exactly(BACKENDS[name]):
        job = device.submit(Program(format='qasm3', data=garnet_verbatim(qc, physical)),
                            shots=shots, tags=QUEST_JOB_TAGS)
    else:
        job = device.run(qc, shots=shots, tags=QUEST_JOB_TAGS)
    counts = job.result().data.get_counts()
    ran_as_written = check_what_ran(job, qc)
    return counts, ran_as_written is not False


Qubit count, vendor, status and queue depth: enough to decide whether a job will run, but nothing about whether it will run well. There are no per-qubit error rates here.

qBraid does publish calibration summaries for some devices, but combining them across vendors is hard. A superconducting chip reports two-qubit gate error for specific pairs of qubits; a trapped-ion device with every qubit connected has no pairs to report and different dominant errors. A single number averaged across both would not compare like with like.

## Mirror circuits

A benchmark needs a circuit whose correct output is known without simulating it, since simulation becomes impossible at useful sizes. Mirror circuits provide this. Run a circuit $C$, then $C^{\dagger}$. Together they are the identity, so the input comes back out, and any deviation is error. The ideal output is known at any width and depth, at no classical cost.

Run as-is, this overestimates quality. Systematic (coherent) errors, such as a pulse that over-rotates, tend to cancel when the circuit is inverted: an over-rotation in $C$ becomes an equal and opposite one in $C^{\dagger}$.

The fix is a random Pauli $P$ between the two halves. Because $C$ is a Clifford circuit, $C^{\dagger} P C$ is another Pauli $P'$, so the ideal output is still a single basis state, just not all zeros. We compute which one classically, which is cheap. The random Pauli stops coherent errors from cancelling, and averaging over many random choices turns them into an effective random error that the decay curve can measure. This is a simplified version of the randomized mirror circuits of Proctor and co-workers.

In [ ]:
CLIFF1Q = ['id', 'x', 'y', 'z', 'h', 's', 'sdg']


def mirror_circuit(nq, depth, rng, pairs=None, qubits=None):
    """
    A randomized mirror circuit on `nq` qubits.

    Returns (circuit, ideal_bitstring). The circuit is C, then a random Pauli,
    then C-dagger. Because C is Clifford, the net operation is a Pauli, so the
    ideal output is one basis state, computed here rather than simulated.
    """
    qubits = list(range(nq)) if qubits is None else list(qubits)

    front = QuantumCircuit(nq)
    for _ in range(depth):
        for q in qubits:
            getattr(front, rng.choice(CLIFF1Q))(q)
        front.barrier()                       # keep the layers from merging
        if pairs:
            for a, b in pairs:
                front.cx(a, b)
            front.barrier()

    z_part = rng.integers(0, 2, nq).astype(bool)
    x_part = rng.integers(0, 2, nq).astype(bool)
    pauli = Pauli((z_part, x_part))

    qc = QuantumCircuit(nq, nq)
    qc.compose(front, inplace=True)
    qc.barrier()
    # Apply the Pauli as explicit gates. Pauli.to_instruction() returns an
    # opaque instruction whose name is the Pauli label ('ZXYX'); the simulator
    # decomposes it, but QASM export keeps that name and the converter rejects
    # it as an undefined gate, so hardware submission fails. Same unitary here,
    # up to a global phase that measurement cannot see.
    for q in range(nq):
        if z_part[q] and x_part[q]:
            qc.y(q)
        elif x_part[q]:
            qc.x(q)
        elif z_part[q]:
            qc.z(q)
    qc.barrier()
    qc.compose(front.inverse(), inplace=True)
    qc.measure(range(nq), range(nq))

    # Heisenberg frame: P -> C-dagger P C, which is what acts on |0...0>.
    net = pauli.evolve(Clifford(front), frame='h')
    ideal = ''.join('1' if b else '0' for b in reversed(net.x))
    return qc, ideal


qc_demo, ideal_demo = mirror_circuit(3, 2, np.random.default_rng(0), pairs=[(0, 1)])
print(f"predicted ideal output: {ideal_demo}")
qc_demo.draw('mpl', fold=120)

In [ ]:
# The prediction is a claim about the circuit. Check it before relying on it.
ok = 0
trials = 40
for _ in range(trials):
    nq = int(rng.integers(1, 6))
    pairs = [(i, i + 1) for i in range(nq - 1)] if nq > 1 else None
    qc, ideal = mirror_circuit(nq, int(rng.integers(1, 8)), rng, pairs)
    counts = sim.run(transpile(qc, sim, optimization_level=0), shots=200).result().get_counts()
    ok += (len(counts) == 1 and list(counts)[0] == ideal)

print(f"predicted output matched noiseless simulation in {ok}/{trials} random instances")

## The compiler can erase the benchmark

A mirror circuit is a circuit followed by its own inverse, which is exactly the pattern an optimizing compiler removes. Left alone, the transpiler recognises the structure and sends the device a few gates instead of a few hundred. The device then reports an excellent result, because it was asked to do almost nothing.

The barriers inside `mirror_circuit` stop Qiskit's transpiler from doing this. The next cell shows the size of the effect.

In [ ]:
def cx_after_transpile(qc, level, barriers=True):
    if not barriers:
        qc = qc.copy()
        qc.data = [d for d in qc.data if d.operation.name != 'barrier']
    return transpile(qc, basis_gates=['cx', 'rz', 'sx', 'x'],
                     optimization_level=level, seed_transpiler=1).count_ops().get('cx', 0)


rows = []
for depth in (2, 4, 8, 16):
    qc, _ = mirror_circuit(4, depth, np.random.default_rng(depth), pairs=[(0, 1), (2, 3)])
    rows.append({
        'depth': depth,
        'CX as written': qc.count_ops().get('cx', 0),
        'CX, barriers, opt 3': cx_after_transpile(qc, 3, barriers=True),
        'CX, no barriers, opt 3': cx_after_transpile(qc, 3, barriers=False),
    })

pd.DataFrame(rows).set_index('depth')

With barriers, Qiskit keeps the full circuit at every depth. Without them, it collapses each mirrored block and the depth sweep becomes flat. A benchmark like that measures the compiler instead of the device, and it fails without any warning, because the numbers look good.

Barriers only instruct Qiskit, though. After qBraid submits a circuit, the device's own compiler processes it again. In our tests it ignored the barriers: on IQM Garnet and on Rigetti Cepheus, mirror circuits like these compiled to no two-qubit gates at all, and both devices reported near-perfect survival.

So the hardware cells below run the circuits *verbatim* on IQM Garnet: translated into Garnet's native gates (`prx` and `cz`) on physical qubits we name, and executed gate for gate. After every job, `check_what_ran` compares the gates we sent with the gates the device ran. Rigetti offers no verbatim mode through qBraid, so it is left out of the hardware sections; one of the exercises shows what happens if you include it.

When running real work you want the opposite: every optimisation the compiler can make. The compilation notebook in this series looks at that side.

## Does the benchmark measure what it claims?

Before trusting a hardware number, we test the method on a simulator with an error rate we choose. We add a known two-qubit depolarizing error, run the sweep, fit the decay, and check that the fit returns the value we put in.

The survival probability of a mirror circuit falls exponentially with the number of noisy gates, down to a floor of $2^{-n}$ where the output is completely random:

$$S(d) = A\, f^{\,g(d)} + 2^{-n},$$

where $g(d)$ is the number of gates at depth $d$. For a two-qubit depolarizing channel of strength $p$, the quantity recovered is the average gate infidelity $3p/4$ rather than $p$, because a fully depolarized output still matches the correct state part of the time. This factor matters if you want to compare with a quoted error rate.

In [ ]:
def survival(circuit_fn, backend, depths, instances, shots, nq):
    """Average survival probability over random mirror instances at each depth."""
    out = []
    for d in depths:
        tot = 0.0
        for _ in range(instances):
            qc, ideal = circuit_fn(d)
            counts = backend.run(transpile(qc, backend, optimization_level=0),
                                 shots=shots).result().get_counts()
            tot += counts.get(ideal, 0) / shots
        out.append(tot / instances)
    return np.array(out)


def fit_infidelity(depths, surv, nq, gates_per_depth):
    """Fit S(d) = A f^g(d) + 2^-n and return the per-gate infidelity 1 - f."""
    y = surv - 2.0 ** -nq
    keep = y > 1e-3
    if keep.sum() < 2:
        return np.nan
    slope = np.polyfit(np.asarray(depths)[keep] * gates_per_depth,
                       np.log(y[keep]), 1)[0]
    return 1 - np.exp(slope)


PAIRS = [(0, 1), (2, 3)]
DEPTHS = [1, 2, 4, 8, 16]
NQ = 4
GATES_PER_DEPTH = 2 * len(PAIRS)          # each layer appears twice, once mirrored

rows = []
for p_inj in (0.005, 0.02, 0.05):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p_inj, 2), ['cx'])
    noisy = AerSimulator(noise_model=nm, seed_simulator=2)

    surv = survival(lambda d: mirror_circuit(NQ, d, rng, PAIRS),
                    noisy, DEPTHS, instances=20, shots=400, nq=NQ)
    est = fit_infidelity(DEPTHS, surv, NQ, GATES_PER_DEPTH)
    rows.append({'injected p': p_inj, 'expected 3p/4': round(3 * p_inj / 4, 4),
                 'recovered': round(est, 4), 'ratio': round(est / (3 * p_inj / 4), 2),
                 'survival': [round(s, 3) for s in surv]})

pd.DataFrame(rows).set_index('injected p')

The fit should recover the injected rate to within a few percent over a tenfold range of error rates. The remaining bias comes from the simple two-parameter fit, and from attributing all of the decay to two-qubit gates when single-qubit gates also contribute. That bias does not matter for ranking, since it affects everything being ranked equally. It does matter for quoting an absolute fidelity, which needs single- and two-qubit contributions fitted separately.

## The same measurement on real devices

The sweep now runs on IQM Garnet, verbatim, on four connected physical qubits (14, 15, 16 and 17). Because we choose the qubits and the device runs our gates exactly, the result is an error per two-qubit layer on those qubits, with no routing added. Devices without a verbatim mode are skipped, with a message.

The default run is 8 jobs and may take minutes to hours depending on queues.

In [ ]:
HW_DEPTHS = [1, 2, 4, 8]
HW_INSTANCES = 2

hw_survival = {}
for name, device in devices.items():
    if not runs_exactly(BACKENDS[name]):
        print(f"{name}: skipped. Its compiler would remove the mirror circuits (see above).")
        continue
    surv = []
    for d in HW_DEPTHS:
        tot = 0.0
        for _ in range(HW_INSTANCES):
            qc, ideal = mirror_circuit(NQ, d, rng, PAIRS)
            counts, _ = run_exactly(name, device, qc, HW_SHOTS, physical=GARNET_PATH[:NQ])
            tot += counts.get(ideal, 0) / HW_SHOTS
        surv.append(tot / HW_INSTANCES)
    hw_survival[name] = np.array(surv)
    print(f"{name:18s} survival {[round(s, 3) for s in surv]}")


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.5))

xs = np.array(HW_DEPTHS) * GATES_PER_DEPTH
ax.axhline(2.0 ** -NQ, color='gray', linestyle=':', linewidth=2,
           label=f'Fully randomised ($2^{{-{NQ}}}$)')
ax.axhline(1.0, color='k', linestyle='--', linewidth=2, alpha=0.6, label='Ideal')

for name in hw_survival:
    ax.plot(xs, hw_survival[name], 'o-', color=COLORS[name], label=name,
            markersize=11, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('two-qubit gates in the circuit')
ax.set_ylabel('survival probability')
ax.set_title('Randomized mirror circuits, run exactly as written')
ax.set_ylim(0, 1.08)
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

summary = pd.DataFrame([
    {'Backend': name,
     'Survival at depth 1': f'{hw_survival[name][0]:.3f}',
     'Survival at depth 8': f'{hw_survival[name][-1]:.3f}',
     'Infidelity per 2Q gate':
         f'{fit_infidelity(HW_DEPTHS, hw_survival[name], NQ, GATES_PER_DEPTH):.4f}'}
    for name in hw_survival
]).set_index('Backend')
summary

This gives one number per device, measured the same way on each, on circuits we chose. It does not replace the vendor's own characterisation, which is more detailed. Its advantage is that it can be compared across vendors.

## Ranking the qubits of one device

A single number per device hides the spread across the chip. Single-qubit mirror circuits reveal it cheaply: one circuit runs an independent random mirror on every qubit at once, and each qubit's survival probability is read from the same results. With no two-qubit gates, there is no routing and no dependence on the layout. On Garnet we rank the eight qubits of the line used above, and the verbatim program keeps each circuit on exactly the qubits we name.

In [ ]:
# Ranking needs a device that runs circuits exactly; otherwise the compiler removes the mirrors.
RANK_TARGET = next(name for name, dev_id in BACKENDS.items() if runs_exactly(dev_id))
RANK_DEPTHS = [4, 16, 64]
RANK_INSTANCES = 2
MAX_RANK_WIDTH = 8                  # cap the ranking circuits: a 108-qubit
                                    # device would otherwise build 108-qubit
                                    # mirror circuits at every depth


def per_qubit_survival(counts, ideal, shots, width):
    """Survival for each qubit separately, from one shot record."""
    hits = np.zeros(width)
    for bits, num in counts.items():
        b = bits.replace(' ', '')
        for q in range(width):
            if b[width - 1 - q] == ideal[width - 1 - q]:
                hits[q] += num
    return hits / shots

In [ ]:
width = min(len(GARNET_PATH), MAX_RANK_WIDTH)
phys = np.array(GARNET_PATH[:width])
print(f"{RANK_TARGET}: ranking physical qubits {phys.tolist()}")

qubit_scores = np.zeros(width)
n_meas = 0

for d in RANK_DEPTHS:
    for _ in range(RANK_INSTANCES):
        qc, ideal = mirror_circuit(width, d, rng, pairs=None)
        counts, _ = run_exactly(RANK_TARGET, devices[RANK_TARGET], qc, RANK_SHOTS, physical=phys)
        qubit_scores += per_qubit_survival(counts, ideal, RANK_SHOTS, width)
        n_meas += 1

qubit_scores /= n_meas
order = np.argsort(-qubit_scores)
print(f"best 4 qubits:  {phys[order[:4]].tolist()}  survival {qubit_scores[order[:4]].round(3).tolist()}")
print(f"worst 4 qubits: {phys[order[-4:]].tolist()}  survival {qubit_scores[order[-4:]].round(3).tolist()}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

colors = ['#2d7a4f' if q in order[:4] else '#a02580' if q in order[-4:] else '#1a5285'
          for q in range(width)]
ax.bar(range(width), qubit_scores, color=colors, width=0.7)
ax.axhline(qubit_scores.mean(), color='k', linestyle='--', linewidth=2, alpha=0.6,
           label=f'device mean ({qubit_scores.mean():.3f})')
ax.set_xlabel('physical qubit index')
ax.set_ylabel('mean survival')
ax.set_title(f'Per-qubit survival on {RANK_TARGET}: green best four, magenta worst four')
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3, axis='y')
ax.legend()
plt.tight_layout()
plt.show()

## What the ranking is worth

A ranking matters only if acting on it changes a result. We take the BB84 circuit from the Cryptography notebook, which prepares the four protocol states on four qubits and measures each in its own basis, and run it twice on the same device in the same session: once on the four best qubits and once on the four worst. Both runs are verbatim, so the device cannot move the circuit to other qubits. The circuit has no two-qubit gates, so placement is the only difference.

It returns the protocol's error rate. BB84 aborts above 11%.

In [ ]:
Z_BASIS, X_BASIS = 0, 1
BB84_STATES = [(0, Z_BASIS), (1, Z_BASIS), (0, X_BASIS), (1, X_BASIS)]


def bb84_on_qubits(physical, width):
    """The four BB84 states, one per chosen physical qubit, each measured in its own basis."""
    qc = QuantumCircuit(width, len(BB84_STATES))
    for slot, (bit, basis) in enumerate(BB84_STATES):
        q = int(physical[slot])
        if bit:
            qc.x(q)
        if basis == X_BASIS:
            qc.h(q)
    qc.barrier()
    # ProtoQuil requires ALL gates before ALL measurements. Interleaving them
    # ('rotate, measure, rotate, measure') fails on Rigetti with
    # "Misplaced or illegal instruction in ProtoQuil program". Rotate every
    # qubit into its measurement basis first, then measure them all.
    for slot, (bit, basis) in enumerate(BB84_STATES):
        if basis == X_BASIS:
            qc.h(int(physical[slot]))
    qc.barrier()
    for slot, _ in enumerate(BB84_STATES):
        qc.measure(int(physical[slot]), slot)
    return qc


def qber_from(counts, shots):
    errs = 0
    for bits, num in counts.items():
        b = bits.replace(' ', '')
        for slot, (bit, _) in enumerate(BB84_STATES):
            if int(b[len(BB84_STATES) - 1 - slot]) != bit:
                errs += num
    return errs / (shots * len(BB84_STATES))


# Sanity check on the simulator: the ideal QBER must be zero for any placement.
for placement in (order[:4], order[-4:]):
    qc = bb84_on_qubits(placement, width)
    counts = sim.run(transpile(qc, sim, optimization_level=0), shots=2000).result().get_counts()
    print(f"ideal QBER on qubits {placement.tolist()}: {qber_from(counts, 2000):.4f}")

In [ ]:
placements = {'best four': order[:4], 'worst four': order[-4:]}

payoff = {}
for label, placement in placements.items():
    qc = bb84_on_qubits(placement, width)
    counts, _ = run_exactly(RANK_TARGET, devices[RANK_TARGET], qc, PAYOFF_SHOTS, physical=phys)
    payoff[label] = qber_from(counts, PAYOFF_SHOTS)
    print(f"{label:11s} qubits {phys[placement].tolist()}  QBER = {payoff[label]:.4f}")


In [ ]:
def key_rate(q):
    """Shor-Preskill asymptotic rate r = 1 - 2h(Q), floored at zero."""
    if q <= 0:
        return 1.0
    if q >= 0.5:
        return 0.0
    h = -q * np.log2(q) - (1 - q) * np.log2(1 - q)
    return max(0.0, 1 - 2 * h)


rows = []
for label, placement in placements.items():
    q = payoff[label]
    rows.append({
        'Placement': label,
        'Qubits': str(placement.tolist()),
        'Mean mirror survival': f'{qubit_scores[placement].mean():.3f}',
        'Measured QBER': f'{q:.4f}',
        'Key rate': f'{key_rate(q):.3f}',
        'Secure bits per 1000 sifted': f'{1000 * key_rate(q):.0f}',
        'BB84 verdict': 'ABORT' if q >= 0.11 else 'ok',
    })

pd.DataFrame(rows).set_index('Placement')

Same device, circuit, session and shot count. The only difference is which four physical qubits ran it, and that choice came from a ranking that took six jobs to produce.

In our tests on IQM Garnet (100 shots), the ranking clearly separated the qubits: survival 0.98 for the best and 0.79 for the worst. The BB84 payoff was smaller: a QBER of 1.0% on the best four qubits and 1.75% on the worst four. Both are far below the 11% threshold, so on this device the choice of qubits did not decide whether the protocol could produce a key. On a device with a wider spread in qubit quality it can, and a report that says only "BB84 on device X" leaves out that variable.

## What this measurement does not tell you

Mirror circuits are cheap, portable and self-checking, which makes them easy to over-interpret. Four limits:

**They measure random circuits, not yours.** The survival probability is an average over random Clifford layers. A specific algorithm with structured errors may behave differently. Mirror circuits are best for comparing placements and devices, as here. Predicting the fidelity of a particular algorithm needs a benchmark shaped like that algorithm.

**They combine everything into one number.** Gate error, readout error, idle decoherence and crosstalk all lower survival, and the fit cannot separate them. A qubit that ranks poorly might have good gates and poor readout. That matters if you want to fix the qubit, not if you only want to avoid it.

**Parallel mirrors measure each qubit with its neighbours active.** This makes the ranking affordable, and it means crosstalk is included in the score. That usually matches how a real circuit runs, but it is not the same as an isolated per-qubit fidelity.

**Rankings go out of date.** Devices are recalibrated regularly and drift in between. A ranking measured hours earlier may no longer hold, which is why the comparison above runs the ranking and the protocol in the same session.

The device is part of the experiment. It varies across the chip and over time, and choosing which qubits to use is part of the method. The QAOA and VQE notebooks use whatever placement the compiler chooses; benchmarking first and pinning the best qubits is one way they could be improved.

## Going further

- **Separate the error sources.** Rerun the depth sweep without the two-qubit layers, so only single-qubit gates and idle time contribute, and subtract. Compare the two-qubit infidelity from this method with the single-fit number above.
- **Measure readout separately.** Prepare each basis state and measure it straight away, with no mirror. Build each qubit's confusion matrix and check how much of the ranking comes from readout rather than gates.
- **Measure crosstalk.** Run single-qubit mirrors on one qubit alone, then on all qubits together, and compare. The difference is what the neighbours cost.
- **Rank pairs of qubits.** Extend the ranking to two-qubit mirrors on candidate pairs, then place a two-qubit circuit on the best and worst pair. Report the transpiled two-qubit gate count too, since routing may differ between the pairs.
- **Track drift.** Run the ranking once an hour for a day and plot how the top four change. How old can a ranking be before you should rerun it?
- **Use the ranking.** In the QAOA or VQE notebook, benchmark first, set `initial_layout` to the best qubits, and check whether the final result improves.